# 10 Map S12 holdings to WFICN

目标：精确连接S12日期映射，并仅按唯一最大type1_assets选择代表报告。

- 输入：09持仓；02 S12映射
- 输出：selected holdings、选择决策、未匹配和资产核验审计
- 主键：wficn+rdate+fundno+cusip8
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：02、09
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
hold_path=upstream_path(ROOT,RUN_MODE,"09_clean_s12_holdings_sources","data/interim/09_s12/s12_holdings_with_rdate.parquet")
link_path=upstream_path(ROOT,RUN_MODE,"02_clean_identifier_links","data/interim/02_links/fundno_wficn_clean.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")
con.execute(f"""CREATE TEMP VIEW mapped_all AS
 SELECT h.*,l.wficn,l.mflink_assets
 FROM read_parquet('{esc(hold_path)}') h
 LEFT JOIN read_parquet('{esc(link_path)}') l
 USING(fundno,fdate,rdate)""")
con.execute("""CREATE TEMP VIEW mapped AS
 SELECT * FROM mapped_all WHERE wficn IS NOT NULL""")
con.execute("""CREATE TEMP TABLE report AS
 SELECT wficn,rdate,fundno,
        first(type1_assets) type1_assets,
        first(mflink_assets) mflink_assets,
        count(*) n_holdings
 FROM mapped GROUP BY wficn,rdate,fundno""")
con.execute("""CREATE TEMP VIEW report_max AS
 SELECT *,max(type1_assets) OVER(PARTITION BY wficn,rdate) max_type1_assets
 FROM report""")
con.execute("""CREATE TEMP VIEW decisions AS
 SELECT *,
   sum(CASE WHEN type1_assets=max_type1_assets AND type1_assets IS NOT NULL
       THEN 1 ELSE 0 END) OVER(PARTITION BY wficn,rdate) n_max,
   CASE
     WHEN max_type1_assets IS NULL THEN 'exclude_all_type1_assets_missing'
     WHEN sum(CASE WHEN type1_assets=max_type1_assets AND type1_assets IS NOT NULL
              THEN 1 ELSE 0 END) OVER(PARTITION BY wficn,rdate)>1
       THEN 'exclude_tied_max_type1_assets'
     WHEN type1_assets=max_type1_assets THEN 'selected_unique_max_type1_assets'
     ELSE 'not_selected'
   END decision
 FROM report_max""")
con.execute("""CREATE TEMP VIEW selected AS
 SELECT m.* FROM mapped m
 JOIN (
   SELECT wficn,rdate,fundno FROM decisions
   WHERE decision='selected_unique_max_type1_assets'
 ) d USING(wficn,rdate,fundno)""")
out=output_path(ROOT,RUN_MODE,"10_map_s12_holdings_to_wficn","data/interim/10_s12_wficn/wficn_report_holdings_selected.parquet")
audit1=output_path(ROOT,RUN_MODE,"10_map_s12_holdings_to_wficn","data/audit/10_multiple_fundno_decisions.csv")
audit2=output_path(ROOT,RUN_MODE,"10_map_s12_holdings_to_wficn","data/audit/10_unmatched_mflink_s12.csv")
audit3=output_path(ROOT,RUN_MODE,"10_map_s12_holdings_to_wficn","data/audit/type1_vs_mflink_assets_check.csv")
duckdb_copy_parquet_atomic(con,"SELECT * FROM selected",out)
duckdb_copy_csv_atomic(con,"SELECT * FROM decisions",audit1)
duckdb_copy_csv_atomic(con,"SELECT *,'no_exact_mflink_s12' reason_code FROM mapped_all WHERE wficn IS NULL",audit2)
duckdb_copy_csv_atomic(con,"""SELECT *,
 type1_assets-mflink_assets assets_difference,
 type1_assets/mflink_assets assets_ratio,
 type1_assets IS NULL type1_assets_missing,
 mflink_assets IS NULL mflink_assets_missing
 FROM report""",audit3)
hold_count=int(con.execute("SELECT count(*) FROM mapped_all").fetchone()[0])
selected_count=int(con.execute("SELECT count(*) FROM selected").fetchone()[0])
dupes=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM selected
 GROUP BY wficn,rdate,fundno,cusip8 HAVING count(*)>1)""").fetchone()[0])
selected_columns={row[0] for row in con.execute("DESCRIBE selected").fetchall()}
assert "assets" not in selected_columns
assert {"type1_assets","mflink_assets"}.issubset(selected_columns)
con.close()
INPUTS_USED=["s12_holdings_with_rdate.parquet","fundno_wficn_clean.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2,audit3]]
INPUT_COUNTS={"holdings":hold_count}; OUTPUT_COUNTS={"selected_holdings":selected_count}; PRIMARY_KEY=["wficn","rdate","fundno","cusip8"]
DUPLICATE_KEY_COUNT=dupes; EXCLUDED_ROW_COUNT=hold_count-selected_count
EXCEPTION_FILE=str(audit1.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["selection_field == type1_assets","mflink_assets was audit-only.","Full selection used an external-memory DuckDB join and atomic COPY outputs."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "10_map_s12_holdings_to_wficn", stage="10", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
